# MapReduce Example: Calculate Revenue by Product

## Mục tiêu

Ví dụ này mô phỏng cơ chế hoạt động của MapReduce thông qua bài toán:

**Tính tổng doanh thu của từng sản phẩm từ dữ liệu bán hàng.**

Dữ liệu đầu vào:

| Product | Revenue |
|----------|----------|
| Laptop | 1000 |
| Phone | 500 |
| Laptop | 1500 |
| Tablet | 700 |
| Phone | 800 |
| Laptop | 1200 |

MapReduce sẽ thực hiện:

1. Map: Chuyển dữ liệu thành các cặp Key-Value.
2. Shuffle & Sort: Gom các Key giống nhau.
3. Reduce: Tính tổng doanh thu cho từng sản phẩm.

In [1]:
sales_data = [
    "Laptop,1000",
    "Phone,500",
    "Laptop,1500",
    "Tablet,700",
    "Phone,800",
    "Laptop,1200"
]

print("Input Data:")
for row in sales_data:
    print(row)

Input Data:
Laptop,1000
Phone,500
Laptop,1500
Tablet,700
Phone,800
Laptop,1200


## Bước 1: Map Phase

Map đọc từng dòng dữ liệu và chuyển thành các cặp Key-Value.

Trong ví dụ này:

- Key = Product
- Value = Revenue

In [7]:
mapped_data = []

for record in sales_data:
    product, revenue = record.split(",")
    mapped_data.append((product, int(revenue)))

print("Map Output:")
for item in mapped_data:
    print(item)

Map Output:
('Laptop', 1000)
('Phone', 500)
('Laptop', 1500)
('Tablet', 700)
('Phone', 800)
('Laptop', 1200)


## Bước 2: Shuffle & Sort Phase

Đây là bước Hadoop tự động thực hiện.

Mục tiêu là gom tất cả các bản ghi có cùng Key vào một nhóm để chuẩn bị cho bước Reduce.

Ví dụ:

(Laptop,1000)

(Laptop,1500)

(Laptop,1200)

↓

Laptop → [1000,1500,1200]

In [3]:
shuffle_data = {}

for product, revenue in mapped_data:

    if product not in shuffle_data:
        shuffle_data[product] = []

    shuffle_data[product].append(revenue)

print("Shuffle Output:")
for key, value in shuffle_data.items():
    print(key, "->", value)

Shuffle Output:
Laptop -> [1000, 1500, 1200]
Phone -> [500, 800]
Tablet -> [700]


## Bước 3: Reduce Phase

Reduce nhận từng Key cùng danh sách Value tương ứng và thực hiện phép tổng hợp.

Trong bài toán này, phép tổng hợp là tính tổng doanh thu.

In [4]:
reduced_data = {}

for product, revenues in shuffle_data.items():
    reduced_data[product] = sum(revenues)

print("Reduce Output:")
for key, value in reduced_data.items():
    print(key, "->", value)

Reduce Output:
Laptop -> 3700
Phone -> 1300
Tablet -> 700


## Kết quả cuối cùng

Sau khi Reduce hoàn tất, ta thu được tổng doanh thu của từng sản phẩm.

In [5]:
import pandas as pd

result = pd.DataFrame(
    reduced_data.items(),
    columns=["Product", "Total Revenue"]
)

result

,Product,Total Revenue
0,Laptop,3700
1,Phone,1300
2,Tablet,700


## Giải thích cơ chế hoạt động

### Input Data

```text
Laptop,1000
Phone,500
Laptop,1500
Tablet,700
Phone,800
Laptop,1200
```

### Map Phase

```text
(Laptop,1000)
(Phone,500)
(Laptop,1500)
(Tablet,700)
(Phone,800)
(Laptop,1200)
```

### Shuffle & Sort

```text
Laptop -> [1000,1500,1200]

Phone -> [500,800]

Tablet -> [700]
```

### Reduce

```text
Laptop -> 3700

Phone -> 1300

Tablet -> 700
```